# Advanced Problems: Python Iterators and Iterables

This notebook contains advanced practice problems with complete solutions for Python **iterators**, **iterables**, `__iter__`, `__next__`, `StopIteration`, iterator exhaustion, re-iterable containers, sequence fallback with `__getitem__`, lazy processing, and composable iterator utilities.

## Best-practice themes

- An **iterator** implements `__next__()` and `__iter__()` returning itself.
- An **iterable** implements `__iter__()` and should normally return a **fresh iterator** each time.
- Avoid storing iteration state on reusable collections.
- Design mutation behavior intentionally: live view, snapshot, or fail-fast.
- Prefer lazy iteration for large or infinite data.
- Add small tests with `assert` to document expected behavior.


In [1]:
from collections import deque
from dataclasses import dataclass
from itertools import islice, count
from typing import Iterable, Iterator, Callable, TypeVar, Generic, Any

T = TypeVar("T")
U = TypeVar("U")

def show(title, value):
    print(f"{title}: {value}")


## Problem 1 — Diagnose iterable vs iterator objects

Write two functions:

1. `is_iterable(obj)` returns `True` when `iter(obj)` succeeds.
2. `is_iterator(obj)` returns `True` when:
   - `iter(obj)` succeeds,
   - `next(obj)` would be valid in principle,
   - and `iter(obj) is obj`.

Then test these against lists, list iterators, dictionaries, dictionary views, generators, strings, and integers.

### Why this matters

Many bugs happen because a function accidentally stores an **iterator** when it should store an **iterable**. The iterator is then consumed once and silently becomes empty.


In [2]:
# Solution 1

def is_iterable(obj: Any) -> bool:
    try:
        iter(obj)
    except TypeError:
        return False
    return True


def is_iterator(obj: Any) -> bool:
    try:
        iterator = iter(obj)
    except TypeError:
        return False
    return iterator is obj and hasattr(obj, "__next__")


samples = {
    "list": [1, 2, 3],
    "list_iterator": iter([1, 2, 3]),
    "dict": {"a": 1, "b": 2},
    "dict_keys_view": {"a": 1, "b": 2}.keys(),
    "generator": (x * x for x in range(3)),
    "string": "abc",
    "integer": 42,
}

for name, obj in samples.items():
    print(f"{name:15} iterable={is_iterable(obj)!s:5} iterator={is_iterator(obj)!s:5}")

assert is_iterable([1, 2, 3]) is True
assert is_iterator([1, 2, 3]) is False

it = iter([1, 2, 3])
assert is_iterable(it) is True
assert is_iterator(it) is True

assert is_iterable(42) is False
assert is_iterator(42) is False


list            iterable=True  iterator=False
list_iterator   iterable=True  iterator=True 
dict            iterable=True  iterator=False
dict_keys_view  iterable=True  iterator=False
generator       iterable=True  iterator=True 
string          iterable=True  iterator=False
integer         iterable=False iterator=False


## Problem 2 — Fix a broken one-shot collection

The class below is broken as a reusable collection because it stores iteration state on itself.

```python
class BrokenDeck:
    def __init__(self, cards):
        self.cards = list(cards)
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.cards):
            raise StopIteration
        card = self.cards[self.index]
        self.index += 1
        return card
```

Tasks:

1. Explain why this object is an iterator, not just an iterable.
2. Rewrite it as a reusable `Deck` iterable.
3. Support two independent iterators over the same deck.
4. Prove with tests that nested loops work correctly.


In [3]:
# Solution 2

class Deck:
    # A reusable iterable collection.
    #
    # Best practice:
    # - The Deck object stores only the data.
    # - Each call to iter(deck) returns a fresh independent iterator.

    def __init__(self, cards: Iterable[str]):
        self._cards = tuple(cards)

    def __len__(self) -> int:
        return len(self._cards)

    def __iter__(self) -> Iterator[str]:
        return self._DeckIterator(self)

    class _DeckIterator:
        def __init__(self, deck: "Deck"):
            self._deck = deck
            self._index = 0

        def __iter__(self) -> "Deck._DeckIterator":
            return self

        def __next__(self) -> str:
            if self._index >= len(self._deck._cards):
                raise StopIteration
            card = self._deck._cards[self._index]
            self._index += 1
            return card


deck = Deck(["A♠", "K♠", "Q♠"])

assert list(deck) == ["A♠", "K♠", "Q♠"]
assert list(deck) == ["A♠", "K♠", "Q♠"]  # reusable

it1 = iter(deck)
it2 = iter(deck)

assert next(it1) == "A♠"
assert next(it1) == "K♠"
assert next(it2) == "A♠"  # independent iterator

pairs = [(left, right) for left in deck for right in deck]
assert pairs == [
    ("A♠", "A♠"), ("A♠", "K♠"), ("A♠", "Q♠"),
    ("K♠", "A♠"), ("K♠", "K♠"), ("K♠", "Q♠"),
    ("Q♠", "A♠"), ("Q♠", "K♠"), ("Q♠", "Q♠"),
]

print("Deck passes reusable, independent, and nested-loop tests.")


Deck passes reusable, independent, and nested-loop tests.


## Problem 3 — Sequence + iterable: implement a read-only catalog

Create a `ProductCatalog` class that supports:

- `len(catalog)`
- indexing: `catalog[0]`
- slicing: `catalog[1:3]`
- iteration
- membership testing by SKU: `"SKU-001" in catalog`
- `catalog.find(sku)`

Use a small immutable `Product` dataclass.

### Best practice requirement

Iteration should return a fresh iterator. The catalog should not expose its internal list directly.


In [4]:
# Solution 3

@dataclass(frozen=True)
class Product:
    sku: str
    name: str
    price: float


class ProductCatalog:
    def __init__(self, products: Iterable[Product]):
        self._products = tuple(products)
        self._by_sku = {product.sku: product for product in self._products}

    def __len__(self) -> int:
        return len(self._products)

    def __getitem__(self, index):
        return self._products[index]

    def __iter__(self) -> Iterator[Product]:
        return iter(self._products)

    def __contains__(self, sku: str) -> bool:
        return sku in self._by_sku

    def find(self, sku: str) -> Product:
        try:
            return self._by_sku[sku]
        except KeyError as exc:
            raise KeyError(f"Unknown SKU: {sku!r}") from exc


catalog = ProductCatalog([
    Product("SKU-001", "Keyboard", 99.0),
    Product("SKU-002", "Mouse", 49.0),
    Product("SKU-003", "Monitor", 249.0),
])

assert len(catalog) == 3
assert catalog[0].name == "Keyboard"
assert [p.sku for p in catalog[1:]] == ["SKU-002", "SKU-003"]
assert [p.name for p in catalog] == ["Keyboard", "Mouse", "Monitor"]
assert [p.price for p in catalog] == [99.0, 49.0, 249.0]
assert "SKU-002" in catalog
assert "SKU-999" not in catalog
assert catalog.find("SKU-003").price == 249.0

print("ProductCatalog behaves as both a sequence and a reusable iterable.")


ProductCatalog behaves as both a sequence and a reusable iterable.


## Problem 4 — Snapshot iteration vs live iteration

Create a mutable `Playlist` class.

Requirements:

- `playlist.add(song)` appends a song.
- `playlist.remove(song)` removes a song.
- `for song in playlist:` should iterate over a **snapshot** of the songs as they were when iteration began.
- Mutating the playlist during a loop should not affect that loop.
- A later loop should see the updated playlist.

### Design note

Snapshot iteration is often safer than live iteration for mutable collections because it avoids surprising skips, duplicates, or infinite loops.


In [5]:
# Solution 4

class Playlist:
    def __init__(self, songs: Iterable[str] = ()): 
        self._songs = list(songs)

    def add(self, song: str) -> None:
        self._songs.append(song)

    def remove(self, song: str) -> None:
        self._songs.remove(song)

    def __len__(self) -> int:
        return len(self._songs)

    def __iter__(self) -> Iterator[str]:
        # Snapshot at iteration start.
        return iter(tuple(self._songs))


playlist = Playlist(["Intro", "Verse", "Chorus"])

seen = []
for song in playlist:
    seen.append(song)
    if song == "Verse":
        playlist.add("Bridge")

assert seen == ["Intro", "Verse", "Chorus"]
assert list(playlist) == ["Intro", "Verse", "Chorus", "Bridge"]

playlist.remove("Intro")
assert list(playlist) == ["Verse", "Chorus", "Bridge"]

print("Snapshot iteration worked.")


Snapshot iteration worked.


## Problem 5 — Fail-fast iteration for mutation detection

Sometimes snapshot iteration hides bugs. Implement a `FailFastList` that raises a `RuntimeError` if the collection is structurally modified during iteration.

Requirements:

- `append(value)` mutates the collection.
- `remove(value)` mutates the collection.
- `__iter__()` returns a custom iterator.
- The iterator stores the collection's version number at creation.
- If the collection's version changes before `__next__`, raise `RuntimeError`.

This mirrors the idea used by many collection libraries: detect dangerous mutation during iteration.


In [6]:
# Solution 5

class FailFastList(Generic[T]):
    def __init__(self, values: Iterable[T] = ()): 
        self._values = list(values)
        self._version = 0

    def append(self, value: T) -> None:
        self._values.append(value)
        self._version += 1

    def remove(self, value: T) -> None:
        self._values.remove(value)
        self._version += 1

    def __len__(self) -> int:
        return len(self._values)

    def __iter__(self) -> Iterator[T]:
        return self._Iterator(self)

    class _Iterator(Iterator[T]):
        def __init__(self, owner: "FailFastList[T]"):
            self._owner = owner
            self._expected_version = owner._version
            self._index = 0

        def __iter__(self) -> "FailFastList._Iterator":
            return self

        def __next__(self) -> T:
            if self._expected_version != self._owner._version:
                raise RuntimeError("FailFastList changed during iteration")

            if self._index >= len(self._owner._values):
                raise StopIteration

            value = self._owner._values[self._index]
            self._index += 1
            return value


numbers = FailFastList([1, 2, 3])
assert list(numbers) == [1, 2, 3]

it = iter(numbers)
assert next(it) == 1
numbers.append(4)

try:
    next(it)
    raise AssertionError("Expected RuntimeError")
except RuntimeError as exc:
    assert "changed during iteration" in str(exc)

assert list(numbers) == [1, 2, 3, 4]

print("Fail-fast iterator detected mutation.")


Fail-fast iterator detected mutation.


## Problem 6 — Lazy line parser with robust error handling

Build a lazy iterable called `SensorLog`.

Each line has this format:

```text
timestamp,sensor_id,value
```

Example:

```text
2026-01-01T00:00:00,A,10.5
```

Requirements:

- The class accepts any iterable of strings.
- It parses lazily: do not convert all lines to a list.
- It skips blank lines.
- It skips a header row if present.
- It yields `Reading(timestamp, sensor_id, value)`.
- Bad rows should either:
  - be skipped when `strict=False`, or
  - raise `ValueError` when `strict=True`.

### Advanced point

This object is reusable only if the input source is reusable. A list of lines is reusable; a file object or generator is usually one-shot.


In [7]:
# Solution 6

@dataclass(frozen=True)
class Reading:
    timestamp: str
    sensor_id: str
    value: float


class SensorLog:
    def __init__(self, lines: Iterable[str], *, strict: bool = False):
        self._lines = lines
        self._strict = strict

    def __iter__(self) -> Iterator[Reading]:
        for line_number, raw_line in enumerate(self._lines, start=1):
            line = raw_line.strip()

            if not line:
                continue

            lowered = line.lower().replace(" ", "")
            if lowered == "timestamp,sensor_id,value":
                continue

            parts = [part.strip() for part in line.split(",")]
            if len(parts) != 3:
                if self._strict:
                    raise ValueError(f"Line {line_number}: expected 3 fields, got {len(parts)}")
                continue

            timestamp, sensor_id, value_text = parts

            try:
                value = float(value_text)
            except ValueError:
                if self._strict:
                    raise ValueError(f"Line {line_number}: invalid numeric value {value_text!r}")
                continue

            yield Reading(timestamp, sensor_id, value)


raw_lines = [
    "timestamp,sensor_id,value",
    "2026-01-01T00:00:00,A,10.5",
    "",
    "bad,row",
    "2026-01-01T00:01:00,B,not-a-number",
    "2026-01-01T00:02:00,C,12.75",
]

log = SensorLog(raw_lines)
assert list(log) == [
    Reading("2026-01-01T00:00:00", "A", 10.5),
    Reading("2026-01-01T00:02:00", "C", 12.75),
]

strict_log = SensorLog(raw_lines, strict=True)
try:
    list(strict_log)
    raise AssertionError("Expected ValueError")
except ValueError as exc:
    assert "Line" in str(exc)

print("SensorLog parses valid rows lazily and handles bad rows correctly.")


SensorLog parses valid rows lazily and handles bad rows correctly.


## Problem 7 — Understand one-shot sources

Use the `SensorLog` from Problem 6 with a one-shot generator.

Tasks:

1. Create a generator that yields log lines.
2. Pass it to `SensorLog`.
3. Show that the first `list(log)` works.
4. Show that the second `list(log)` is empty.
5. Fix the issue by passing a reusable container instead.

This problem demonstrates a key design fact: a reusable iterable cannot magically make a one-shot input reusable unless it caches the data.


In [8]:
# Solution 7

def make_log_line_generator():
    yield "timestamp,sensor_id,value"
    yield "2026-01-01T00:00:00,A,10.5"
    yield "2026-01-01T00:01:00,B,11.5"


one_shot_lines = make_log_line_generator()
log_from_generator = SensorLog(one_shot_lines)

first_pass = list(log_from_generator)
second_pass = list(log_from_generator)

assert first_pass == [
    Reading("2026-01-01T00:00:00", "A", 10.5),
    Reading("2026-01-01T00:01:00", "B", 11.5),
]
assert second_pass == []

# Fix: store the lines in a reusable container when the data size is reasonable.
reusable_lines = list(make_log_line_generator())
reusable_log = SensorLog(reusable_lines)

assert list(reusable_log) == first_pass
assert list(reusable_log) == first_pass

print("One-shot source behavior demonstrated and fixed.")


One-shot source behavior demonstrated and fixed.


## Problem 8 — Implement a `Peekable` iterator

Create a `Peekable` wrapper around any iterable.

Requirements:

- `peek()` returns the next item without consuming it.
- `next(peekable)` consumes and returns the next item.
- Multiple `peek()` calls should return the same item.
- `bool(peekable)` returns `True` if at least one item remains.
- It must work when the next value is `None`.

### Best practice

Never use `None` as an internal sentinel when `None` is a valid data value. Use a unique sentinel object.


In [9]:
# Solution 8

_MISSING = object()


class Peekable(Iterator[T]):
    def __init__(self, iterable: Iterable[T]):
        self._iterator = iter(iterable)
        self._cache: Any = _MISSING

    def __iter__(self) -> "Peekable[T]":
        return self

    def _fill_cache(self) -> None:
        if self._cache is _MISSING:
            self._cache = next(self._iterator)

    def peek(self) -> T:
        self._fill_cache()
        return self._cache

    def __next__(self) -> T:
        if self._cache is not _MISSING:
            value = self._cache
            self._cache = _MISSING
            return value

        return next(self._iterator)

    def __bool__(self) -> bool:
        try:
            self._fill_cache()
        except StopIteration:
            return False
        return True


p = Peekable([None, "A", "B"])

assert bool(p) is True
assert p.peek() is None
assert p.peek() is None
assert next(p) is None
assert p.peek() == "A"
assert next(p) == "A"
assert next(p) == "B"
assert bool(p) is False

try:
    p.peek()
    raise AssertionError("Expected StopIteration")
except StopIteration:
    pass

print("Peekable works, including when the next item is None.")


Peekable works, including when the next item is None.


## Problem 9 — Implement lazy `take`, `drop`, and `tabulate`

Create three iterator utilities:

1. `take(n, iterable)` yields at most the first `n` items.
2. `drop(n, iterable)` skips the first `n` items and yields the rest.
3. `tabulate(function, start=0)` yields `function(start)`, `function(start + 1)`, ... forever.

Requirements:

- All utilities must be lazy.
- `take` must be able to stop an infinite iterable.
- `drop` must not build an intermediate list.


In [10]:
# Solution 9

def take(n: int, iterable: Iterable[T]) -> Iterator[T]:
    if n < 0:
        raise ValueError("n must be non-negative")
    iterator = iter(iterable)
    for _ in range(n):
        try:
            yield next(iterator)
        except StopIteration:
            return


def drop(n: int, iterable: Iterable[T]) -> Iterator[T]:
    if n < 0:
        raise ValueError("n must be non-negative")
    iterator = iter(iterable)
    for _ in range(n):
        try:
            next(iterator)
        except StopIteration:
            return
    yield from iterator


def tabulate(function: Callable[[int], T], start: int = 0) -> Iterator[T]:
    index = start
    while True:
        yield function(index)
        index += 1


assert list(take(3, [10, 20, 30, 40])) == [10, 20, 30]
assert list(take(10, [1, 2])) == [1, 2]
assert list(drop(2, [10, 20, 30, 40])) == [30, 40]
assert list(drop(10, [1, 2])) == []
assert list(take(5, tabulate(lambda x: x * x))) == [0, 1, 4, 9, 16]
assert list(take(4, tabulate(lambda x: x + 100, start=5))) == [105, 106, 107, 108]

print("take, drop, and tabulate are lazy and tested.")


take, drop, and tabulate are lazy and tested.


## Problem 10 — Sliding windows over an iterable

Implement `sliding_window(iterable, size)`.

Examples:

```python
list(sliding_window([1, 2, 3, 4], 2))
# [(1, 2), (2, 3), (3, 4)]

list(sliding_window([1, 2, 3, 4], 3))
# [(1, 2, 3), (2, 3, 4)]
```

Requirements:

- Return tuples.
- Work lazily.
- Do not materialize the full input.
- Raise `ValueError` if `size <= 0`.
- If the iterable has fewer than `size` items, yield nothing.


In [11]:
# Solution 10

def sliding_window(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)
    window = deque(maxlen=size)

    for _ in range(size):
        try:
            window.append(next(iterator))
        except StopIteration:
            return

    yield tuple(window)

    for item in iterator:
        window.append(item)
        yield tuple(window)


assert list(sliding_window([1, 2, 3, 4], 2)) == [(1, 2), (2, 3), (3, 4)]
assert list(sliding_window([1, 2, 3, 4], 3)) == [(1, 2, 3), (2, 3, 4)]
assert list(sliding_window([1, 2], 3)) == []
assert list(take(3, sliding_window(count(1), 4))) == [
    (1, 2, 3, 4),
    (2, 3, 4, 5),
    (3, 4, 5, 6),
]

try:
    list(sliding_window([1, 2, 3], 0))
    raise AssertionError("Expected ValueError")
except ValueError:
    pass

print("sliding_window works for finite and infinite inputs.")


sliding_window works for finite and infinite inputs.


## Problem 11 — Batch an iterable

Implement `batched(iterable, batch_size, *, strict=False)`.

Examples:

```python
list(batched([1, 2, 3, 4, 5], 2))
# [(1, 2), (3, 4), (5,)]

list(batched([1, 2, 3, 4], 2, strict=True))
# [(1, 2), (3, 4)]

list(batched([1, 2, 3], 2, strict=True))
# ValueError
```

Requirements:

- Lazy.
- No full materialization.
- Return tuples.
- Reject non-positive batch sizes.


In [12]:
# Solution 11

def batched(iterable: Iterable[T], batch_size: int, *, strict: bool = False) -> Iterator[tuple[T, ...]]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    iterator = iter(iterable)

    while True:
        batch = tuple(islice(iterator, batch_size))

        if not batch:
            return

        if strict and len(batch) != batch_size:
            raise ValueError("incomplete final batch")

        yield batch


assert list(batched([1, 2, 3, 4, 5], 2)) == [(1, 2), (3, 4), (5,)]
assert list(batched([1, 2, 3, 4], 2, strict=True)) == [(1, 2), (3, 4)]
assert list(take(3, batched(count(1), 4))) == [
    (1, 2, 3, 4),
    (5, 6, 7, 8),
    (9, 10, 11, 12),
]

try:
    list(batched([1, 2, 3], 2, strict=True))
    raise AssertionError("Expected ValueError")
except ValueError as exc:
    assert "incomplete" in str(exc)

print("batched is lazy and supports strict mode.")


batched is lazy and supports strict mode.


## Problem 12 — Round-robin iteration

Implement `round_robin(*iterables)`.

Example:

```python
list(round_robin("ABC", [1, 2], ("x", "y", "z", "w")))
# ["A", 1, "x", "B", 2, "y", "C", "z", "w"]
```

Requirements:

- Work with iterables of different lengths.
- Remove exhausted iterators.
- Preserve cyclic order among remaining iterators.
- Be lazy.


In [13]:
# Solution 12

def round_robin(*iterables: Iterable[T]) -> Iterator[T]:
    active = deque(iter(it) for it in iterables)

    while active:
        iterator = active.popleft()

        try:
            value = next(iterator)
        except StopIteration:
            continue

        yield value
        active.append(iterator)


assert list(round_robin("ABC", [1, 2], ("x", "y", "z", "w"))) == [
    "A", 1, "x",
    "B", 2, "y",
    "C", "z",
    "w",
]

assert list(round_robin([], [1, 2, 3])) == [1, 2, 3]
assert list(take(7, round_robin(count(1), "abc"))) == [1, "a", 2, "b", 3, "c", 4]

print("round_robin handles uneven and infinite inputs lazily.")


round_robin handles uneven and infinite inputs lazily.


## Problem 13 — Recursive flattening without flattening strings

Implement `flatten(iterable)`.

Requirements:

- Flatten nested iterables recursively.
- Do **not** split strings or bytes into characters.
- Work with lists, tuples, generators, sets, and other iterables.
- Preserve iteration order for ordered collections.

Example:

```python
list(flatten([1, [2, 3], (4, [5, "abc"]), b"xy"]))
# [1, 2, 3, 4, 5, "abc", b"xy"]
```


In [14]:
# Solution 13

_ATOMIC_TYPES = (str, bytes, bytearray)


def flatten(iterable: Iterable[Any]) -> Iterator[Any]:
    for item in iterable:
        if isinstance(item, _ATOMIC_TYPES):
            yield item
            continue

        try:
            nested_iterator = iter(item)
        except TypeError:
            yield item
        else:
            yield from flatten(nested_iterator)


nested = [1, [2, 3], (4, [5, "abc"]), b"xy"]
assert list(flatten(nested)) == [1, 2, 3, 4, 5, "abc", b"xy"]

def nested_generator():
    yield 10
    yield [20, (30, 40)]
    yield "done"

assert list(flatten(nested_generator())) == [10, 20, 30, 40, "done"]

print("flatten recursively handles nested iterables and preserves strings/bytes.")


flatten recursively handles nested iterables and preserves strings/bytes.


## Problem 14 — Breadth-first traversal as an iterator

Implement a reusable `Graph` iterable whose iteration performs breadth-first search from a starting node.

Requirements:

- The graph stores an adjacency dictionary.
- `graph.bfs(start)` returns an iterator over nodes.
- It must not loop forever on cycles.
- Traversal should be lazy.
- Neighbor order should follow the order in the adjacency list.


In [15]:
# Solution 14

class Graph:
    def __init__(self, adjacency: dict[str, list[str]]):
        self._adjacency = {node: tuple(neighbors) for node, neighbors in adjacency.items()}

    def neighbors(self, node: str) -> tuple[str, ...]:
        return self._adjacency.get(node, ())

    def bfs(self, start: str) -> Iterator[str]:
        return self._BFSIterator(self, start)

    class _BFSIterator(Iterator[str]):
        def __init__(self, graph: "Graph", start: str):
            self._graph = graph
            self._queue = deque([start])
            self._seen = {start}

        def __iter__(self) -> "Graph._BFSIterator":
            return self

        def __next__(self) -> str:
            if not self._queue:
                raise StopIteration

            node = self._queue.popleft()

            for neighbor in self._graph.neighbors(node):
                if neighbor not in self._seen:
                    self._seen.add(neighbor)
                    self._queue.append(neighbor)

            return node


graph = Graph({
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": ["A"],  # cycle back to A
    "E": [],
    "F": ["B"],  # cycle back to B
})

assert list(graph.bfs("A")) == ["A", "B", "C", "D", "E", "F"]
assert list(graph.bfs("C")) == ["C", "F", "B", "D", "E", "A"]

it = graph.bfs("A")
assert next(it) == "A"
assert next(it) == "B"

print("Graph BFS is lazy and cycle-safe.")


Graph BFS is lazy and cycle-safe.


## Problem 15 — Build a reusable lazy pipeline

Create a `Pipeline` class that can chain transformations while staying reusable.

Required methods:

- `Pipeline(iterable)`
- `.map(function)`
- `.filter(predicate)`
- `.take(n)`
- iteration over the pipeline

Example:

```python
pipeline = (
    Pipeline(range(100))
    .filter(lambda x: x % 2 == 0)
    .map(lambda x: x * 10)
    .take(5)
)

list(pipeline)
# [0, 20, 40, 60, 80]
```

### Best-practice challenge

Do not store an already-created iterator inside `Pipeline`, or the pipeline will become one-shot. Store the original iterable and a list of operations instead.


In [16]:
# Solution 15

class Pipeline(Generic[T]):
    def __init__(self, iterable: Iterable[T], operations: tuple[tuple[str, Any], ...] = ()): 
        self._iterable = iterable
        self._operations = operations

    def map(self, function: Callable[[Any], Any]) -> "Pipeline":
        return Pipeline(self._iterable, self._operations + (("map", function),))

    def filter(self, predicate: Callable[[Any], bool]) -> "Pipeline":
        return Pipeline(self._iterable, self._operations + (("filter", predicate),))

    def take(self, n: int) -> "Pipeline":
        if n < 0:
            raise ValueError("n must be non-negative")
        return Pipeline(self._iterable, self._operations + (("take", n),))

    def __iter__(self) -> Iterator[Any]:
        iterator = iter(self._iterable)

        for operation, argument in self._operations:
            if operation == "map":
                iterator = map(argument, iterator)
            elif operation == "filter":
                iterator = filter(argument, iterator)
            elif operation == "take":
                iterator = islice(iterator, argument)
            else:
                raise RuntimeError(f"Unknown operation: {operation!r}")

        yield from iterator


pipeline = (
    Pipeline(range(100))
    .filter(lambda x: x % 2 == 0)
    .map(lambda x: x * 10)
    .take(5)
)

assert list(pipeline) == [0, 20, 40, 60, 80]
assert list(pipeline) == [0, 20, 40, 60, 80]  # reusable because range is reusable

# Works lazily with infinite sources.
infinite_pipeline = (
    Pipeline(count(1))
    .filter(lambda x: x % 3 == 0)
    .map(lambda x: x ** 2)
    .take(4)
)

assert list(infinite_pipeline) == [9, 36, 81, 144]

print("Pipeline is reusable for reusable sources and lazy for infinite sources.")


Pipeline is reusable for reusable sources and lazy for infinite sources.


## Problem 16 — Caching iterable: make a one-shot source reusable

Create `CachedIterable`, a wrapper that accepts any iterable, including a one-shot generator, and makes repeated iteration possible by caching values as they are produced.

Requirements:

- First pass pulls from the source and stores values.
- Later passes replay cached values and continue reading new values if needed.
- Multiple iterators over the same cache should work independently.
- It should be lazy: do not consume the entire source upfront.

### Warning

Caching infinite sources can grow memory forever. This is powerful, but it must be used deliberately.


In [17]:
# Solution 16

class CachedIterable(Generic[T]):
    def __init__(self, source: Iterable[T]):
        self._source_iterator = iter(source)
        self._cache: list[T] = []
        self._exhausted = False

    def __iter__(self) -> Iterator[T]:
        index = 0

        while True:
            if index < len(self._cache):
                yield self._cache[index]
                index += 1
                continue

            if self._exhausted:
                return

            try:
                value = next(self._source_iterator)
            except StopIteration:
                self._exhausted = True
                return

            self._cache.append(value)
            index += 1
            yield value


def noisy_source():
    for value in [10, 20, 30]:
        print(f"pulling {value}")
        yield value


cached = CachedIterable(noisy_source())

it1 = iter(cached)
it2 = iter(cached)

assert next(it1) == 10  # pulls 10
assert next(it1) == 20  # pulls 20

assert next(it2) == 10  # replays from cache
assert next(it2) == 20  # replays from cache
assert next(it2) == 30  # pulls 30

assert list(cached) == [10, 20, 30]
assert list(cached) == [10, 20, 30]

print("CachedIterable lazily turns a one-shot source into a reusable iterable.")


pulling 10
pulling 20
pulling 30
CachedIterable lazily turns a one-shot source into a reusable iterable.


## Problem 17 — Iterator protocol interview challenge

Predict the output before running this cell.

Questions:

1. Why does `list(a)` return values only once?
2. Why does `list(b)` return values every time?
3. Why does `iter(a) is a` evaluate to `True`?
4. Why does `iter(b) is b` evaluate to `False`?


In [18]:
# Solution 17

a = iter([1, 2, 3])
b = [1, 2, 3]

print("iter(a) is a:", iter(a) is a)
print("iter(b) is b:", iter(b) is b)

print("first list(a):", list(a))
print("second list(a):", list(a))

print("first list(b):", list(b))
print("second list(b):", list(b))

assert iter(a) is a
assert iter(b) is not b

# a was already consumed by the first list(a)
assert list(a) == []

# b is a reusable iterable
assert list(b) == [1, 2, 3]

print("Iterator exhaustion and reusable iterable behavior confirmed.")


iter(a) is a: True
iter(b) is b: False
first list(a): [1, 2, 3]
second list(a): []
first list(b): [1, 2, 3]
second list(b): [1, 2, 3]
Iterator exhaustion and reusable iterable behavior confirmed.


## Problem 18 — Build a custom reverse iterator without using `reversed`

Create `ReverseView`.

Requirements:

- Accept any finite iterable.
- Store a snapshot as a tuple.
- Iteration yields values in reverse order.
- The view is reusable.
- Do not call Python's built-in `reversed()` in your `__iter__`.

This is a useful exercise for understanding manual iterator state.


In [19]:
# Solution 18

class ReverseView(Generic[T]):
    def __init__(self, iterable: Iterable[T]):
        self._items = tuple(iterable)

    def __len__(self) -> int:
        return len(self._items)

    def __iter__(self) -> Iterator[T]:
        return self._ReverseIterator(self._items)

    class _ReverseIterator(Iterator[T]):
        def __init__(self, items: tuple[T, ...]):
            self._items = items
            self._index = len(items) - 1

        def __iter__(self) -> "ReverseView._ReverseIterator":
            return self

        def __next__(self) -> T:
            if self._index < 0:
                raise StopIteration

            value = self._items[self._index]
            self._index -= 1
            return value


view = ReverseView(["a", "b", "c"])

assert list(view) == ["c", "b", "a"]
assert list(view) == ["c", "b", "a"]

it = iter(view)
assert next(it) == "c"
assert next(it) == "b"
assert next(it) == "a"

try:
    next(it)
    raise AssertionError("Expected StopIteration")
except StopIteration:
    pass

print("ReverseView uses a custom reusable reverse iterator.")


ReverseView uses a custom reusable reverse iterator.


## Problem 19 — Mini project: iterable leaderboard

Create a `Leaderboard` class for game scores.

Requirements:

- `add(player, score)` adds a new score.
- Iterating over the leaderboard yields entries sorted by:
  1. highest score first,
  2. earliest insertion first for ties.
- `top(n)` returns the first `n` entries.
- Iteration must be snapshot-based.
- Adding a score during iteration should not affect the active iterator.


In [20]:
# Solution 19

@dataclass(frozen=True)
class ScoreEntry:
    player: str
    score: int
    order: int


class Leaderboard:
    def __init__(self):
        self._entries: list[ScoreEntry] = []
        self._next_order = 0

    def add(self, player: str, score: int) -> None:
        self._entries.append(ScoreEntry(player, score, self._next_order))
        self._next_order += 1

    def __len__(self) -> int:
        return len(self._entries)

    def __iter__(self) -> Iterator[ScoreEntry]:
        snapshot = tuple(self._entries)
        ordered = sorted(snapshot, key=lambda entry: (-entry.score, entry.order))
        return iter(ordered)

    def top(self, n: int) -> list[ScoreEntry]:
        if n < 0:
            raise ValueError("n must be non-negative")
        return list(islice(iter(self), n))


board = Leaderboard()
board.add("Ada", 100)
board.add("Linus", 120)
board.add("Grace", 120)
board.add("Guido", 90)

assert [(entry.player, entry.score) for entry in board] == [
    ("Linus", 120),
    ("Grace", 120),
    ("Ada", 100),
    ("Guido", 90),
]

assert [(entry.player, entry.score) for entry in board.top(2)] == [
    ("Linus", 120),
    ("Grace", 120),
]

iterator = iter(board)
first = next(iterator)
board.add("New Champion", 999)

# The existing iterator uses a snapshot.
remaining_players = [entry.player for entry in iterator]
assert first.player == "Linus"
assert "New Champion" not in remaining_players

# A new iteration sees the new entry.
assert next(iter(board)).player == "New Champion"

print("Leaderboard implements snapshot-based sorted iteration.")


Leaderboard implements snapshot-based sorted iteration.


## Problem 20 — Comprehensive debugging exercise

The following class looks reasonable, but it has a subtle bug:

```python
class BadRecentItems:
    def __init__(self, items):
        self.items = items
        self.iterator = iter(items)

    def __iter__(self):
        return self.iterator
```

Tasks:

1. Explain the bug.
2. Rewrite it as `RecentItems`.
3. Add `.latest(n)` that returns the last `n` items in reverse chronological order.
4. Ensure `RecentItems` is reusable.


In [21]:
# Solution 20

class RecentItems(Generic[T]):
    def __init__(self, items: Iterable[T]):
        # Store a reusable snapshot. If items is already a list, this also protects
        # us from external mutation.
        self._items = tuple(items)

    def __len__(self) -> int:
        return len(self._items)

    def __iter__(self) -> Iterator[T]:
        # Fresh iterator every time.
        return iter(self._items)

    def latest(self, n: int) -> list[T]:
        if n < 0:
            raise ValueError("n must be non-negative")

        result = []
        index = len(self._items) - 1

        while index >= 0 and len(result) < n:
            result.append(self._items[index])
            index -= 1

        return result


recent = RecentItems(["v1", "v2", "v3", "v4"])

assert list(recent) == ["v1", "v2", "v3", "v4"]
assert list(recent) == ["v1", "v2", "v3", "v4"]
assert recent.latest(2) == ["v4", "v3"]
assert recent.latest(10) == ["v4", "v3", "v2", "v1"]

source = ["a", "b"]
snapshot = RecentItems(source)
source.append("c")

assert list(snapshot) == ["a", "b"]

print("RecentItems fixes the stored-iterator bug.")


RecentItems fixes the stored-iterator bug.


# Final checklist

Use this checklist when designing custom iteration behavior:

- Does the object represent a reusable collection? Then `__iter__` should usually return a **new iterator**.
- Does the object represent a stream or cursor? Then it can be its own iterator.
- Does `__next__` raise `StopIteration` exactly when exhausted?
- Can two iterators over the same iterable progress independently?
- Is mutation behavior intentional: snapshot, live, or fail-fast?
- Are strings/bytes handled carefully when recursively processing iterables?
- Are large or infinite inputs processed lazily?
- Are tests included for repeated iteration, nested iteration, and exhaustion?
